# 2a - From lists to arrays

**Time:** 45 minutes  |  **AI: explains** (Gemini may explain an error or a line to you)

**What you will do**
- Meet NumPy: arrays, and calculations on all values at once.
- Compute the return and the risk of a portfolio of three funds, A, B and C.
- Simulate 5,000 possible years for one fund, and see why arrays are fast.
- For each cell: write your prediction first, then run it, then compare.

How to work: click a cell, press Shift+Enter to run it. Run the cells in order, top to bottom. Nothing fails on purpose in this notebook.

## Catch the error (AI off, in pairs)

Two short pieces of code from real finance work, the same as on the slides. Each one runs without an error message, and each one gives a wrong answer. Run it, look at the result, find the problem, then fix it in the cell below it.

### Catch the error 1 - Long A and C, short B. What is the risk?

Vols 15%, 20%, 25%. Weights 100%, -100%, 100%. The model uses these correlations:

| | A | B | C |
|---|---|---|---|
| A | 1.0 | 0.9 | -0.9 |
| B | 0.9 | 1.0 | 0.9 |
| C | -0.9 | 0.9 | 1.0 |

Run the cell. You don't need to read every line yet: you will learn them all today. Look at the two results.

In [ ]:
import numpy as np

vol = np.array([0.15, 0.20, 0.25])     # funds A, B, C
w = np.array([1.0, -1.0, 1.0])         # long A, short B, long C
corr = np.array([[1.0, 0.9, -0.9],
                 [0.9, 1.0, 0.9],
                 [-0.9, 0.9, 1.0]])

cov = np.diag(vol) @ corr @ np.diag(vol)
variance = w @ cov @ w
print("variance =", round(variance, 4))
print("volatility =", np.sqrt(variance))

A variance below zero? A volatility that is `nan` ("not a number": the square root of a negative)? Look at the correlations: A with B, B with C, A with C. Can all three be true at the same time?

Fix it in the cell below: change the A-C correlation (it appears twice) to a value that fits the other two. Check: with 0.7, the variance is **0.0335** and the volatility **18.3%**.

In [ ]:
# Fix the A-C correlation (two places), then run
import numpy as np

vol = np.array([0.15, 0.20, 0.25])     # funds A, B, C
w = np.array([1.0, -1.0, 1.0])         # long A, short B, long C
corr = np.array([[1.0, 0.9, -0.9],
                 [0.9, 1.0, 0.9],
                 [-0.9, 0.9, 1.0]])

cov = np.diag(vol) @ corr @ np.diag(vol)
variance = w @ cov @ w
print("variance =", round(variance, 4))
print("volatility =", np.sqrt(variance))

### Catch the error 2 - Same code, three runs. Which number?

Run the cell below **three times** (Shift+Enter, three times) and write down the three numbers.

In [ ]:
import numpy as np

rng = np.random.default_rng()          # random numbers
daily = rng.normal(0.07 / 252, 0.25 / 252 ** 0.5, size=(252, 5000))
final = 100 * np.prod(1 + daily, axis=0)
print("Chance of a loss:", (final < 100).mean())

Which number goes in the email to the MD? Fix the code so that every run, and every student, gets the same number. Hint: look inside the brackets of `default_rng()`, and use 42. Check: you should get **0.435** every time.

In [ ]:
# Fix it so every run gives the same number, then run it three times
import numpy as np

rng = np.random.default_rng()          # random numbers
daily = rng.normal(0.07 / 252, 0.25 / 252 ** 0.5, size=(252, 5000))
final = 100 * np.prod(1 + daily, axis=0)
print("Chance of a loss:", (final < 100).mean())

## The story

A client holds three funds. The MD asks: what is the risk of this portfolio?

| | Fund A | Fund B | Fund C |
|---|---|---|---|
| Expected return | 5% | 7% | 9% |
| Volatility (risk) | 15% | 20% | 25% |
| Weight in the client portfolio | 20% | 30% | 50% |

Correlations: A-B 0.1, A-C 0.4, B-C 0.7. Rates and weights are decimals: 7% is `0.07`.

## Cell 1 - Double every weight: list or array?

In Session 1 you used lists. NumPy adds the **array**. `import numpy as np` loads the toolbox and calls it `np`.

**Your prediction:** what does `[0.2, 0.3, 0.5] * 2` give? And the same with an array?

In [ ]:
w_list = [0.2, 0.3, 0.5]   # a list, as in Session 1
print(w_list * 2)

import numpy as np          # load the NumPy toolbox, call it np
w = np.array([0.2, 0.3, 0.5])   # an array: numbers you can compute with at once
print(w * 2)

## Cell 2 - Portfolio expected return

Multiply each weight by its fund's return (a vectorized operation), then add everything with `np.sum`. These are the client weights: 20 / 30 / 50.

You should get **7.6%**.

**Your prediction:** will the portfolio return be above 7%, below 7%, or exactly 7%? (Look at the weights: which fund is the biggest?)

In [ ]:
ret = np.array([0.05, 0.07, 0.09])   # expected returns of A, B, C

contributions = w * ret              # each fund's contribution, value by value
print(contributions)

port_ret = np.sum(w * ret)           # add them up
print(f"Portfolio return: {port_ret:.1%}")

## Cell 3 - The same in one operator: `@`

`w @ ret` multiplies value by value and adds everything up. It is the **matrix product** (here the dot product of two arrays). It does cell 2 in one step.

You should get **7.6%** again.

**Your prediction:** the same number as cell 2?

In [ ]:
print(f"Portfolio return with @: {w @ ret:.1%}")

## Cell 4 - The correlation matrix: a 2D array

A table of numbers is a **2D array**: a list of rows. `corr.shape` gives (rows, columns). `corr[0, 2]` is row 0, column 2: positions start at 0, like in a list.

**Your prediction:** what is `corr[0, 2]`? (A is row 0, C is column 2.)

In [ ]:
vol = np.array([0.15, 0.20, 0.25])   # volatilities of A, B, C
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])

print(corr.shape)     # (rows, columns)
print(corr[0, 2])     # row 0, column 2: correlation of A and C
print(corr[1, 2])     # correlation of B and C

## Cell 5 - The covariance matrix: vols and correlations together

The covariance of two funds is `vol_i * corr_ij * vol_j`. `np.diag(vol)` is a **diagonal matrix**: the vols on the diagonal and 0 everywhere else. Multiplying `diag(vol) @ corr @ diag(vol)` does every pair at once.

`np.round(cov, 4)` shows 4 decimals.

You should get `cov[0, 0]` = **0.0225**.

**Your prediction:** what is `cov[0, 0]`? (Hint: the correlation of a fund with itself is 1, and the vol of A is 15%.)

In [ ]:
cov = np.diag(vol) @ corr @ np.diag(vol)   # every pair at once
print(np.round(cov, 4))

print(f"cov[0, 0] = {cov[0, 0]:.4f}")
print(f"15% squared = {0.15 ** 2:.4f}")

## Cell 6 - Risk of an equal-weight portfolio

The portfolio **variance** is `w @ cov @ w`. The **volatility** is its square root: `np.sqrt`. Here the three funds have equal weights, one third each. `np.ones(3)` makes `[1, 1, 1]`.

You should get **16.02%**, against 20.00% for the average of the three vols.

**Your prediction:** is the portfolio vol above, below or equal to the average vol (20%)?

In [ ]:
eq = np.ones(3) / 3                   # equal weights: 1/3 each

var_p = eq @ cov @ eq                 # portfolio variance
vol_p = np.sqrt(var_p)                # square root: variance -> volatility
avg_vol = np.sum(eq * vol)            # weighted average of the vols

print(f"Portfolio vol: {vol_p:.2%}")  # :.2% shows a decimal as a percent
print(f"Average vol:   {avg_vol:.2%}")

## Cell 7 - Random numbers: one year of daily returns for fund C

Fund C: expected return 7% a year, volatility 25% a year, 252 trading days. A daily return is random: its mean is `0.07 / 252` and its spread is `0.25 / 252 ** 0.5`.

`np.random.default_rng(42)` makes a **random generator**. The number 42 is the **seed**: the same seed gives the same random numbers, every time, for everyone. This is the fix for the "no seed" case at the start of the session.

**Your prediction:** we create the generator a second time with seed 42. Same numbers, or different?

In [ ]:
rng = np.random.default_rng(42)       # random generator, seed 42
daily_1 = rng.normal(0.07 / 252, 0.25 / 252 ** 0.5, size=252)   # 252 daily returns
print(daily_1[:3])

rng = np.random.default_rng(42)       # a new generator, same seed
daily_2 = rng.normal(0.07 / 252, 0.25 / 252 ** 0.5, size=252)
print(daily_2[:3])

print(f"Return over the year: {np.prod(1 + daily_1) - 1:.1%}")   # np.prod multiplies all values

## Cell 8 - 5,000 possible years: a loop, then an array

Now we want 5,000 possible years for fund C, starting at 100. First the way you know from Session 1: a for loop inside a for loop (5,000 paths, 252 days each). We time it with `time.time()`.

**Your prediction:** how long will the loop take? 0.01 seconds, 1 second, or 1 minute?

In [ ]:
import time

rng = np.random.default_rng(42)
start = time.time()

final_loop = []                          # the value after one year, one per path
for path in range(5000):
    value = 100                          # start at 100
    for day in range(252):
        daily_return = rng.normal(0.07 / 252, 0.25 / 252 ** 0.5)   # one random number
        value = value * (1 + daily_return)
    final_loop.append(value)

print(f"Loop: {time.time() - start:.2f} seconds")

The array way: **one line draws all the random numbers**. `size=(252, 5000)` is the **shape**: 252 rows (days) and 5,000 columns (paths). `np.prod(..., axis=0)` multiplies down each column, so one value per path.

We create the generator again with seed 42 right before, so your numbers match the slides.

**Your prediction:** how many times faster will it be?

In [ ]:
rng = np.random.default_rng(42)
start = time.time()

daily = rng.normal(0.07 / 252, 0.25 / 252 ** 0.5, size=(252, 5000))   # days x paths
final = 100 * np.prod(1 + daily, axis=0)                              # value after one year, per path

print(f"Array: {time.time() - start:.3f} seconds")
print(daily.shape)
print(final.shape)

The loop takes about 1 to 2 seconds in Colab (it depends on the machine), the array about 0.01 seconds.

**Careful:** the loop and the array version draw the random numbers in a **different order**, so their 5,000 results are not the same numbers. We only compare the **timing**. From now on we use the array version (`final`), as on the slides.

## Cell 9 - Where could fund C be in a year?

`final` holds 5,000 possible values of 100 invested in fund C, one year later.
- `np.median(final)`: the middle value.
- `np.percentile(final, 5)`: 5% of the paths end below this value. A bad year.
- `final < 100` is an array of True and False, one per path. `.mean()` of True/False is the share of True: the chance of a loss.

You should get median **104.1**, 5th percentile **68.9**, chance of a loss **43.5%**.

**Your prediction:** fund C expects +7% a year. What is the chance that you lose money in a year? 5%, 20% or 40%?

In [ ]:
print(f"Median after a year:        {np.median(final):.1f}")
print(f"Bad year (5th percentile):  {np.percentile(final, 5):.1f}")

lost = final < 100                       # True where the path ended below 100
print(f"Chance of a loss:           {lost.mean():.1%}")

## Cell 10 - See it

50 of the 5,000 paths. `np.cumprod` keeps the running product down each column: the value of the fund day after day. Charts are taught properly in Session 3: here, just read the lines.

**Your prediction:** will all the lines end above 100?

In [ ]:
import matplotlib.pyplot as plt

paths = 100 * np.cumprod(1 + daily[:, :50], axis=0)   # first 50 paths, day by day
plt.plot(paths)
plt.axhline(100, color="black", linestyle="--")       # the starting value
plt.xlabel("Trading days")
plt.ylabel("Value of fund C")
plt.show()

## What you learned

**Python**
- **Import**: `import numpy as np` loads a toolbox. **Array**: `np.array([0.2, 0.3, 0.5])`, numbers you compute with at once.
- **Vectorized operation**: `w * ret` works value by value, with no loop. **Sum**: `np.sum(w * ret)` adds all values.
- **Matrix product**: `w @ ret`, and `w @ cov @ w` for the portfolio variance.
- **2D array + index**: `corr[0, 2]` is row 0, column 2 (from 0). **Shape**: `corr.shape` is `(3, 3)`; `size=(252, 5000)` is days x paths.
- **Diagonal matrix**: `np.diag(vol)` puts the vols on the diagonal. **Square root**: `np.sqrt(var)` turns variance into volatility.
- **Random generator + seed**: `np.random.default_rng(42)`, the same seed gives the same numbers.
- `np.prod(..., axis=0)`, `np.cumprod`, `np.median`, `np.percentile`, and a comparison `final < 100` with `.mean()` to get a share.

**Finance**
- Portfolio return is the weighted average: 7.6%. Portfolio risk is **not**: 16.02% against 20% for equal weights.
- Simulation shows a range, not one number: fund C has a 43.5% chance of a loss in a year.
- Always fix the seed, so the same code gives the same number.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right